# Language Detection with API

In [ ]:
#Upload files
from google.colab import drive
drive.mount('/content/drive')


INPUT_FILE = "/content/drive/MyDrive/file_name.csv"
OUTPUT_FILE = "/content/drive/MyDrive/file_name.csv"
CHECKPOINT_INTERVAL = 10


Mounted at /content/drive


In [ ]:
!pip install openai tqdm

In [ ]:
import pandas as pd
import openai
from collections import Counter
import os
import time
from tqdm import tqdm

In [ ]:
client = openai.OpenAI(
    api_key="your_api_key_here",
    base_url="your_base_url_here"
)

In [ ]:
# LANGUAGE DETECTION
cols_to_check = ["General Comments"] + [
    f"Answer to specific info request {i}" for i in range(1, 11)
]


if os.path.exists(OUTPUT_FILE):
    df = pd.read_csv(OUTPUT_FILE)
    print(" Existing file found. Resuming from previous progress.")
else:
    df = pd.read_csv(INPUT_FILE)
    df["language"] = None

# Language detection function
def detect_language(text):
    if pd.isna(text) or not str(text).strip():
        return None

    prompt = f"""
Detect the language of this text and respond with only the 2-letter ISO 639-1 language code (e.g. en, de, fr, it).

Text:
\"{text.strip()}\"
"""
    try:
        response = client.chat.completions.create(
            model="gpt-35-turbo",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content.strip().lower()
    except Exception as e:
        print(f" Error at row {idx}: {e}")
        return None

# LOOP
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Detecting language"):
    if pd.notna(row["language"]):
        continue

    row_langs = []
    for col in cols_to_check:
        lang = detect_language(row[col])
        if lang:
            row_langs.append(lang)
        time.sleep(0.5)

    dominant_lang = Counter(row_langs).most_common(1)[0][0] if row_langs else None
    df.at[idx, "language"] = dominant_lang


    if idx % 5 == 0:
        df.to_csv(OUTPUT_FILE, index=False)
        print(f" Checkpoint saved at row {idx}")


df.to_csv(OUTPUT_FILE, index=False)
print(" Language detection complete and saved.")


# Translation with API

In [ ]:
# EXTRACT NON EN ROWS

# Load full dataset with language detection
df = pd.read_csv("/content/drive/MyDrive/file_language_detected.csv")


df_to_translate = df[df["language"] != "en"].copy()


df_to_translate.to_csv("/content/drive/MyDrive/file_to_translate.csv", index=False)
print(f"Step 2 complete: {len(df_to_translate)} rows saved for translation.")


✅ Step 2 complete: 1350 rows saved for translation.


In [ ]:
INPUT_TRANSLATE_FILE = "/content/drive/MyDrive/file_to_translate.csv"
OUTPUT_TRANSLATE_FILE = "/content/drive/MyDrive/file_translated_subset.csv"
CHECKPOINT_INTERVAL = 5


client = openai.OpenAI(
    api_key="your_api_key",
    base_url="your_base_url"
)

In [ ]:
# TRANSLATION
cols_to_translate = ["General Comments"] + [
    f"Answer to specific info request {i}" for i in range(1, 11)
]


if os.path.exists(OUTPUT_TRANSLATE_FILE):
    df = pd.read_csv(OUTPUT_TRANSLATE_FILE)
    print(" Resuming from previous checkpoint...")
else:
    df = pd.read_csv(INPUT_TRANSLATE_FILE)
    for col in cols_to_translate:
        df[f"{col}_translated"] = None
        df[f"{col}_translated"] = df[f"{col}_translated"].astype("string")

# Translation function with detected language
def translate_text(text, source_lang):
    if pd.isna(text) or not str(text).strip():
        return text

    prompt = f"""
The original language is {source_lang}. Translate the following text into English. Only return the translated text.

Text:
\"{text.strip()}\"
"""
    try:
        response = client.chat.completions.create(
            model="gpt-35-turbo",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f" Translation error: {e}")
        return text

# LOOP
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Translating rows"):
    if all(pd.notna(row[f"{col}_translated"]) for col in cols_to_translate):
        continue

    source_lang = row["language"]

    for col in cols_to_translate:
        if pd.isna(row[f"{col}_translated"]) and pd.notna(row[col]):
            df.at[idx, f"{col}_translated"] = translate_text(row[col], source_lang)
            time.sleep(0.5)

    if idx % CHECKPOINT_INTERVAL == 0:
        df.to_csv(OUTPUT_TRANSLATE_FILE, index=False)
        print(f"💾 Checkpoint saved at row {idx}")


df.to_csv(OUTPUT_TRANSLATE_FILE, index=False)
print(" Translation completed and saved.")


Translating rows:   0%|          | 1/1350 [00:01<43:51,  1.95s/it]

💾 Checkpoint saved at row 0


Translating rows:   0%|          | 3/1350 [00:03<21:15,  1.06it/s]

💾 Checkpoint saved at row 5


Translating rows:   1%|          | 11/1350 [00:09<19:10,  1.16it/s]

...

💾 Checkpoint saved at row 1340


Translating rows: 100%|█████████▉| 1346/1350 [42:57<00:19,  4.86s/it]

💾 Checkpoint saved at row 1345


Translating rows: 100%|██████████| 1350/1350 [43:04<00:00,  1.91s/it]

✅ Translation completed and saved.


## Extra file cleaning and merging

### Cleaning

In [ ]:
# Save a column with ISO codes
import pandas as pd
import re


df = pd.read_csv("/content/drive/MyDrive/file_language_detected.csv")


def is_iso_639_1(lang):
    return bool(re.fullmatch(r"[a-z]{2}", str(lang).strip().lower()))


df_non_iso = df[~df["language"].apply(is_iso_639_1)].copy()


df_non_iso.to_csv("/content/drive/MyDrive/pfas_non_iso_languages.csv", index=False)
print(f" Subset saved: {len(df_non_iso)} rows with non-ISO languages.")


✅ Subset saved: 472 rows with non-ISO languages.


In [ ]:

full_file = "/content/drive/MyDrive/file_language_detected.csv"
corrections_file = "/content/drive/MyDrive/Copia di pfas_non_iso_languages.xlsx"
output_file = "/content/drive/MyDrive/pfas_comments_with_language_corrected.csv"


df_full = pd.read_csv(full_file)
df_corrections = pd.read_excel(corrections_file)


df_merged = df_full.merge(df_corrections, on="ID", how="left")


df_merged["language_corrected"] = df_merged["actual language"].combine_first(df_merged["language"])


df_merged.drop(columns=["actual language"], inplace=True)


df_merged.to_csv(output_file, index=False)
print(f" Done! Corrected file saved as: {output_file}")


✅ Done! Corrected file saved as: /content/drive/MyDrive/pfas_comments_with_language_corrected.csv


In [ ]:
import pandas as pd


full_file = "/content/drive/MyDrive/pfas_comments_with_language.csv"
corrections_file = "/content/drive/MyDrive/Copia di pfas_non_iso_languages.xlsx"
output_file = "/content/drive/MyDrive/pfas_comments_with_language_corrected.csv"


df_full = pd.read_csv(full_file)
df_corrections = pd.read_excel(corrections_file)


df_merged = df_full.merge(df_corrections, on="ID", how="left")
df_merged["language"] = df_merged["actual language"].combine_first(df_merged["language"])


df_merged.drop(columns=["actual language"], inplace=True)


df_merged.to_csv(output_file, index=False)
print(f"✅ Cleaned file saved to: {output_file}")


✅ Cleaned file saved to: /content/drive/MyDrive/pfas_comments_with_language_corrected.csv


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/pfas_comments_with_language_corrected.csv")


def is_iso_639_1(lang):
    return bool(re.fullmatch(r"[a-z]{2}", str(lang).strip().lower()))


df_non_iso = df[~df["language"].apply(is_iso_639_1)].copy()


df_non_iso.to_csv("/content/drive/MyDrive/pfas_non_iso_languages_corrected.csv", index=False)
print(f" Subset saved: {len(df_non_iso)} rows with non-ISO languages (after correction).")


✅ Subset saved: 349 rows with non-ISO languages (after correction).


In [ ]:

df = pd.read_csv("/content/drive/MyDrive/pfas_comments_with_language_corrected.csv")


manual_fixes = {
    6267: "en",
    6387: "en",
    7706: "en",
    8562: "en",
    9391: "en",
    9559: "en"
}

df["language"] = df.apply(
    lambda row: manual_fixes[row["ID"]] if row["ID"] in manual_fixes else row["language"],
    axis=1
)

def clean_language(lang):
    return lang if re.fullmatch(r"[a-z]{2}", str(lang).strip().lower()) else "undetermined"

df["language"] = df["language"].apply(clean_language)


df.to_csv("/content/drive/MyDrive/pfas_comments_with_language_final.csv", index=False)
print(" Manual fixes applied and non-ISO languages replaced with 'undetermined'.")


✅ Manual fixes applied and non-ISO languages replaced with 'undetermined'.


In [ ]:
df["language"].value_counts()

,count
language,
en,4399
sv,747
undetermined,343
de,93
un,14
ja,11
it,9
fr,8
nl,7


In [ ]:
df_un = df[df["language"] == "un"].copy()
df_un.to_csv("/content/drive/MyDrive/pfas_language_un_to_review.csv", index=False)
print(" Exported 'un' rows for manual review.")


📁 Exported 'un' rows for manual review.


In [ ]:
import pandas as pd


df = pd.read_csv("/content/drive/MyDrive/pfas_comments_with_language_final.csv")


manual_fixes = {
    4943: "en",
    6080: "en",
    6081: "undetermined",
    6625: "en",
    6652: "en",
    6711: "en",
    7098: "undetermined",
    8205: "en",
    8300: "undetermined",
    8555: "en",
    8681: "en",
    8955: "undetermined",
    9128: "de",
    9550: "en"
}


df["language"] = df.apply(
    lambda row: manual_fixes[row["ID"]] if row["ID"] in manual_fixes else row["language"],
    axis=1
)


output_path = "/content/drive/MyDrive/pfas_comments_with_language_final_v2.csv"
df.to_csv(output_path, index=False)
print(f" Manual fixes applied and saved to: {output_path}")


✅ Manual fixes applied and saved to: /content/drive/MyDrive/pfas_comments_with_language_final_v2.csv


In [ ]:
df["language"].value_counts()

,count
language,
en,4408
sv,747
undetermined,347
de,94
ja,11
it,9
fr,8
nl,7
zh,4


In [ ]:
df[df["language"] == "de"]["General Comments"]

,General Comments
54,NaN
58,"Sehr geehrte Damen und Herren, bei einem PFAS-..."
636,NaN
667,Wir verarbeiten PTFE-Schläuche und haben keine...
722,- 4599
...,...
5526,Wir unterstützen entschieden den Vorschlag des...
5563,- 9514
5567,Der gegenwärtige EU PFAS-Verbotsvorschlag besc...
5616,NaN


In [ ]:
import pandas as pd


df = pd.read_csv("/content/drive/MyDrive/pfas_comments_with_language_final_v2.csv")


more_fixes = {
    3863: "undetermined",
    4068: "en",
    4701: "undetermined",
    8707: "en"
}


df["language"] = df.apply(
    lambda row: more_fixes[row["ID"]] if row["ID"] in more_fixes else row["language"],
    axis=1
)


output_path = "/content/drive/MyDrive/pfas_comments_with_language_final_v3.csv"
df.to_csv(output_path, index=False)
print(f" Final fixes applied and saved to: {output_path}")


✅ Final fixes applied and saved to: /content/drive/MyDrive/pfas_comments_with_language_final_v3.csv


In [ ]:
df["language"].value_counts()

,count
language,
en,4410
sv,747
undetermined,349
de,94
ja,10
fr,8
nl,7
it,7
zh,4


### Merging

In [ ]:

language_file = "/content/drive/MyDrive/pfas_comments_with_language_final_v3.csv"
translations_file = "/content/drive/MyDrive/pfas_comments_translated_subset.csv"

df_language = pd.read_csv(language_file)
df_translated = pd.read_csv(translations_file)


translated_cols = ["ID"] + [col for col in df_translated.columns if col.endswith("_translated")]

df_merged = df_language.merge(df_translated[translated_cols], on="ID", how="left")


output_file = "/content/drive/MyDrive/pfas_comments_final_with_translations.csv"
df_merged.to_csv(output_file, index=False)

print(f" Merged dataset saved as: {output_file}")


✅ Merged dataset saved as: /content/drive/MyDrive/pfas_comments_final_with_translations.csv


In [ ]:
import pandas as pd


merged_path = "/content/drive/MyDrive/pfas_comments_final_with_translations.csv"
df = pd.read_csv(merged_path)


original_cols = ["General Comments"] + [
    f"Answer to specific info request {i}" for i in range(1, 11)
]
translated_cols = [col + "_translated" for col in original_cols]


df_english = df.copy()

for orig, trans in zip(original_cols, translated_cols):
    df_english[orig] = df_english[trans].combine_first(df_english[orig])


df_english = df_english.drop(columns=translated_cols)


output_file = "/content/drive/MyDrive/pfas_comments_all_english.csv"
df_english.to_csv(output_file, index=False)

print(f"All-English dataset saved to: {output_file}")


✅ All-English dataset saved to: /content/drive/MyDrive/pfas_comments_all_english.csv
